In [1]:
from pyscripts.AutoEncoder_Experiment import *
import sys, jax, json, os
import jax.numpy as jnp
from datetime import datetime

import pyscripts.jax_amber3 as ja
from pyscripts.Maths import Maths

jax.config.update("jax_enable_x64", True)

In [2]:
#os.environ['XLA_FLAGS'] = ('--xla_gpu_enable_triton_softmax_fusion=true ',
#                           '--xla_gpu_triton_gemm_any=True ',
#                           '--xla_gpu_enable_async_collectives=true ',
#                           '--xla_gpu_enable_latency_hiding_scheduler=true ',
#                           '--xla_gpu_enable_highest_priority_async_stream=true ')

In [3]:
json_fn = 'json_inputs/testingDA.json'
with open(json_fn, 'r') as g:
        json_params = json.load(g)

In [4]:
#Main uses structural loss immediately
#experiment = AutoEncoder_Experiment(json_fn).main()

In [5]:
self = AutoEncoder_Experiment(json_fn)

(8000, 306) (2000, 306)
### MODEL TYPE = ReLu_Dropout ###
######################################
#####     Initializing Done!     #####
AutoEncoder(
    # attributes
    input_size = 306
    n_latents = 17
    hidden_layers = [306, 306, 306]
    activators = [<jax._src.custom_derivatives.custom_jvp object at 0x7fed8f434550>, <jax._src.custom_derivatives.custom_jvp object at 0x7fed8f434550>, <jax._src.custom_derivatives.custom_jvp object at 0x7fed8f434550>, <jax._src.custom_derivatives.custom_jvp object at 0x7fed8f434550>]
    layer_ops = [<class 'flax.linen.linear.Dense'>, <class 'flax.linen.linear.Dense'>, <class 'flax.linen.linear.Dense'>, <class 'flax.linen.linear.Dense'>]
    dropout_rates = [0.2, 0.2, 0.2]
)
######################################


2024-02-23 02:20:39.243689: E external/xla/xla/service/slow_operation_alarm.cc:65] 
********************************
[Compiling module jit_rmsd_distance_matrix] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************
2024-02-23 02:22:24.682192: E external/xla/xla/service/slow_operation_alarm.cc:133] The operation took 3m45.438705905s

********************************
[Compiling module jit_rmsd_distance_matrix] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************


In [ ]:
funcs, weights = [self.math.atom_rmsd], [1]
_ = self.train_to_threshold(funcs, weights, 'rmsd', 0.1, 2000, nan_check_ind=None)
self.plot_losses()

0 6.6452E-01 1.0026E+00 NAN 9.8953E-01 6.6458E-01 1.0029E+00 NAN 9.8930E-01 [1]
1 5.6852E-01 1.0068E+00 NAN 9.8217E-01 5.6840E-01 1.0068E+00 NAN 9.8155E-01 [1]
2 5.5603E-01 9.7902E-01 NAN 9.6039E-01 5.5588E-01 9.7905E-01 NAN 9.5949E-01 [1]
3 5.4853E-01 9.7127E-01 NAN 8.8204E-01 5.4833E-01 9.7127E-01 NAN 8.8044E-01 [1]
4 5.4289E-01 9.8332E-01 NAN 7.5188E-01 5.4263E-01 9.8306E-01 NAN 7.5014E-01 [1]
5 5.3868E-01 9.8276E-01 NAN 6.5175E-01 5.3837E-01 9.8278E-01 NAN 6.5031E-01 [1]
6 5.2268E-01 9.8480E-01 NAN 5.9056E-01 5.2240E-01 9.8460E-01 NAN 5.8931E-01 [1]
7 5.0962E-01 9.7324E-01 NAN 5.5408E-01 5.0936E-01 9.7294E-01 NAN 5.5315E-01 [1]
8 5.0758E-01 9.8028E-01 NAN 5.4007E-01 5.0733E-01 9.7999E-01 NAN 5.3908E-01 [1]
9 5.0648E-01 9.7704E-01 NAN 5.2377E-01 5.0623E-01 9.7683E-01 NAN 5.2278E-01 [1]
10 5.0464E-01 9.7583E-01 NAN 5.1402E-01 5.0440E-01 9.7563E-01 NAN 5.1303E-01 [1]
11 5.0234E-01 9.7487E-01 NAN 5.0670E-01 5.0211E-01 9.7455E-01 NAN 5.0576E-01 [1]
12 5.0189E-01 9.7418E-01 NAN 5.0071E-0

In [ ]:
funcs, weights = [self.math.atom_rmsd, self.math.scaled_pot_enr_diff], [1, 0]
#_ = self.train_nepochs(funcs, weights, 'mean', 10, nan_check_ind=-2)
_ = self.train_scaling_coef(funcs, weights, 'rmsd', -1, 6000)
self.plot_losses()

In [6]:
#Test N_epochs on combination of functions with unequal weights
#    def train_nepochs(self, loss_metrics:list, weights:list, averaging_method:str, num_epochs:int, nan_check_ind=-1):
#self = AutoEncoder_Experiment(json_fn)
funcs, weights = [self.math.atom_rmsd, self.kernel_function], [1, 1]
_ = self.train_nepochs(funcs, weights, 'mean', 200, nan_check_ind=-2)
self.plot_losses()

ConcretizationTypeError: Abstract tracer value encountered where concrete value is expected: traced array with shape int64[].
The size argument of jnp.nonzero must be statically specified to use jnp.nonzero within JAX transformations.
The error occurred while tracing the function loss_function at /media/volume/sdb/githubs/Deep-MMS/pyscripts/Maths.py:169 for jit. This concrete value was not available in Python because it depends on the value of the argument decoded.

See https://jax.readthedocs.io/en/latest/errors.html#jax.errors.ConcretizationTypeError

In [ ]:
#Test Threshold on one function
#train_to_threshold(self, loss_metrics:list, weights:list, averaging_method:str, threshold:float, cutoff_epoch:int, nan_check_ind:int=-1)
#self = AutoEncoder_Experiment(json_fn)
#funcs, weights = [self.math.atom_rmsd], [1]
#_ = self.train_to_threshold(funcs, weights, 'mean', 2, 2000, nan_check_ind=-2)
#self.plot_losses()

In [ ]:
#Test Threshold on multiple functions with unequal weights
#train_to_threshold(self, loss_metrics:list, weights:list, averaging_method:str, threshold:float, cutoff_epoch:int, nan_check_ind:int=-1)
#self = AutoEncoder_Experiment(json_fn)
funcs, weights = [self.math.atom_rmsd, self.math.atom_rmtd], [1, 2]
_ = self.train_to_threshold(funcs, weights, 'mean', 2, 2000, nan_check_ind=-4)
self.plot_losses()

In [ ]:
#Test Scaling
#def train_scaling_coef(self, loss_metrics:list, weights:list, averaging_method:str, scaling_index:int, cutoff_epoch:int, freq:int=10, nan_check_ind:int=-1):
#self = AutoEncoder_Experiment(json_fn)
funcs, weights = [self.math.atom_rmsd, self.math.atom_rmtd, self.math.scaled_pot_enr_diff], [1, 1, 0]
#_ = self.train_nepochs(funcs, weights, 'mean', 10, nan_check_ind=-2)
_ = self.train_scaling_coef(funcs, weights, 'mean', -1, 4000)
self.plot_losses()

In [ ]:
# SCALE IN POTENTIAL ENERGY
print('##############################')
print('#####', f'StSc {self.epoch:05d}', '#####')
print('##############################')
end_scaling_epoch = self.train_scaling_coef(self.math.summation_distance, 'mean', scaling_cutoff)

In [ ]:
# REACH A THRESHOLD OF LOSS AGAIN
print('##############################')
print('#####', f'Scld {self.epoch:05d}', '#####')
print('##############################')
end_training_epoch = self.train_to_threshold(self.math.summation_distance, 'mean', final_thresh, final_cutoff)

In [ ]:
# DONE REPORT THE LAST EPOCH
print('##############################')
print('#####', f'Done {self.epoch:05d}', '#####')
print('##############################')

In [ ]:
rng_init = jax.random.PRNGKey(54)
rng, key = jax.random.split(rng_init)
recon, latents = self.state.apply_fn({'params':self.state.params}, self.test_data, rng)

In [ ]:
for i in range(latents.shape[-1]):
    plt.clf()
    plt.title(f'Latent {i}')
    _ = plt.hist(latents[:, i], bins=100)
    plt.show()

In [ ]:
from sklearn.mixture import GaussianMixture

In [ ]:
ics = []
for i in range(1, 20):
    X = np.array(latents)
    MM = GaussianMixture(n_components=i).fit(X)
    ics.append((i, MM.aic(X), MM.bic(X)))
ics = np.array(ics)
plt.clf()
_ = plt.plot(ics[:, 0], ics[:, 1])
_ = plt.plot(ics[:, 0], ics[:, 2])
plt.legend(('Akaike Info Criterion', 'Bayes Info Criterion'))
plt.xlabel('Num Components')
plt.show()

In [ ]:
X = np.array(latents)
MM = GaussianMixture(n_components=2).fit(X) #chosen based on above graph
samples = MM.sample(600)[0]

In [ ]:
for i in range(samples.shape[-1]):
    plt.clf()
    _ = plt.hist(latents[:,i], bins=100, histtype='step', color='g')
    _ = plt.hist(samples[:,i], bins=100, histtype='step', color='b')
    plt.legend(('Reconstructed from Input', 'Sampled from Mixture Model'))
    plt.title(f'Latent {i+1}')
    plt.show()

In [ ]:
import scipy
n_L = latents.shape[-1]
fig, axs = plt.subplots(n_L, n_L, figsize=(15, 10), sharex='col', sharey='row')
for i in range(n_L):
    for j in range(n_L):
        axs[i,j].scatter(latents[:,j], latents[:,i])
        print(i, j, scipy.stats.pearsonr(latents[:,j], latents[:,i]))
fig.savefig('7L.png', dpi=600)

In [ ]:
gas_fun, tors_fun = ja.get_amber_functions(json_params['files']['fname_prmtop'])

In [ ]:
plt.clf()

recon_energies = gas_fun(recon)

decoded_samples = self.model.apply({'params': self.state.params}, samples, rng, method=self.model.decode)
sampled_energies = gas_fun(decoded_samples)

org_energies = gas_fun(self.test_data)

threshold = 1e3
num_excluded_gmm = len(sampled_energies) - len(sampled_energies[sampled_energies<threshold])
num_excluded_nn  = len(recon_energies) - len(recon_energies[recon_energies<threshold])
print(f'Excluding Outliers GMM - {num_excluded_gmm} RECON - {num_excluded_nn}')

_ = plt.hist(org_energies[org_energies < threshold], bins=100, histtype='step', color='r')
_ = plt.hist(recon_energies[recon_energies<threshold], bins=300, histtype='step', color='g')
_ = plt.hist(sampled_energies[sampled_energies<threshold], bins=300, histtype='step', color='b')

plt.legend(('Test Data Potential', 'Reconstructed from Test', 'Sampled from Mixture Model'))
plt.xlabel('Energy (kJ/mol)')
plt.title('Comparison of Energies - GMM and NN')
plt.show()

In [ ]:
plt.clf()
_ = plt.hist(org_energies, bins=100, histtype='step', color='r')
plt.show()
plt.clf()
_ = plt.hist(recon_energies, bins=100, histtype='step', color='g')
plt.show()
plt.clf()
_ = plt.hist(sampled_energies, bins=100, histtype='step', color='b')
plt.show()

In [ ]:
def write_traj(filename, traj_xyz): #(n conf, n_atoms*3) OR (n conf, n_atoms, 3)
        if traj_xyz.shape[-1] != 3:
            traj_xyz = traj_xyz.reshape(traj_xyz.shape[0], -1, 3)
        with md.formats.DCDTrajectoryFile(filename, 'w') as f:
            f.write(traj_xyz*10) #*10 because mdtraj loads data in nm but saves it in angstrom

In [ ]:
my_dict = {'DA_TestData.dcd' : experiment.test_data, 'DA_ReconData.dcd' : recon, 'DA_GMM.dcd' : decoded_samples}
for key, value in zip(my_dict.keys(), my_dict.values()):
    write_traj(key, value)

In [ ]:
some_vals = [1, 2, 3, 4, 5]

In [ ]:
a = 2

In [ ]:
print(f"vals are {[(val:.2E) for val in some_vals]}")

In [ ]:
print(f"{a:.2E}")